# LogSAD — Setup & Evaluation Results
**Group 6 | Introduction to Deep Learning**

Paper: *Towards Training-free Anomaly Detection with Vision and Language Foundation Models* (CVPR 2025)

---
### Training Status: **Category D — Training-free**
LogSAD uses **pre-trained frozen foundation models** — no gradient updates, no fine-tuning.
- CLIP ViT-L/14 (DataComp-1B)
- DINOv2 ViT-L/14
- SAM ViT-H

The only "setup" steps are:
1. GPT-4V generates Match-of-Thought proposals **offline** (one-time, already done)
2. Memory bank is built from normal reference images at inference time

> **Note on file naming.** LogSAD is **training-free (Category D)**, so there is no fine-tuning notebook.
> This notebook is the training-free equivalent of `GroupX-fulltraining.ipynb`: it shows the **setup (frozen model loading, Match-of-Thought, memory bank)** and the **full evaluation results reproduced vs. the paper**.
> Live inference is in `Group6-demo.ipynb`.

In [1]:
#OWN CODE — run the notebook from the repo root so local modules import correctly
import os, sys
# Run from LogSAD root so all local modules are importable
ROOT = os.path.abspath('..')
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print('Working directory:', os.getcwd())

Working directory: /home/gaya6/LogSAD


## 1. Foundation Model Loading
Three frozen models are loaded at startup. No weights are trained.

In [2]:
#OWN CODE — load the three frozen foundation models and report their parameter counts
import torch
import open_clip_local as open_clip
from dinov2.dinov2.hub.backbones import dinov2_vitl14
from segment_anything import sam_model_registry

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# CLIP ViT-L/14  (DataComp-1B pre-trained)
model_clip, _, _ = open_clip.create_model_and_transforms(
    'hf-hub:laion/CLIP-ViT-L-14-DataComp.XL-s13B-b90K'
)
model_clip.eval()
clip_params = sum(p.numel() for p in model_clip.parameters()) / 1e6
print(f'CLIP ViT-L/14      : {clip_params:.0f}M params  (frozen)')

# DINOv2 ViT-L/14
model_dinov2 = dinov2_vitl14()
model_dinov2.eval()
dino_params = sum(p.numel() for p in model_dinov2.parameters()) / 1e6
print(f'DINOv2 ViT-L/14    : {dino_params:.0f}M params  (frozen)')

# SAM ViT-H
sam = sam_model_registry['vit_h'](checkpoint='./checkpoint/sam_vit_h_4b8939.pth')
sam_params = sum(p.numel() for p in sam.parameters()) / 1e6
print(f'SAM ViT-H          : {sam_params:.0f}M params  (frozen)')

print(f'\nTotal              : {clip_params+dino_params+sam_params:.0f}M params')
print('All models frozen — NO training loop, NO backpropagation.')

/home/gaya6/miniconda3/envs/logsad/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
CLIP ViT-L/14      : 428M params  (frozen)


/home/gaya6/LogSAD/dinov2/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/gaya6/LogSAD/dinov2/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/gaya6/LogSAD/dinov2/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


DINOv2 ViT-L/14    : 304M params  (frozen)
SAM ViT-H          : 641M params  (frozen)

Total              : 1374M params
All models frozen — NO training loop, NO backpropagation.


## 2. Match-of-Thought (MoT) — juice_bottle

GPT-4V was used **offline (one-time)** to generate:
- **Interests of thought** — object parts to segment with SAM+CLIP
- **Compositional rules of thought** — logical constraints to check

This replaces manual annotation of logical rules.

In [3]:
# Transcribed from the paper (Table 5) — Match-of-Thought proposals, not our own design
# Match-of-Thought proposals for juice_bottle (from paper Table 5)
mot_juice_bottle = {
    'interests_of_thought': [
        'glass',
        'liquid in bottle',
        'fruit',
        'label',
        'black background',
    ],
    'compositional_rules_of_thought': [
        'consistency of fruit tag and liquid color'
    ],
    'anomaly_free_caption': [
        'cherry juice bottle (red liquid + cherry tag)',
        'orange juice bottle (yellow liquid + orange tag)',
        'banana juice bottle (milky liquid + banana tag)',
    ],
    'liquid_color_prompts': ['red liquid', 'yellow liquid', 'milky liquid'],
    'fruit_tag_prompts':    ['cherry', 'orange', 'banana'],
}

print('=== juice_bottle Match-of-Thought ===')
for key, val in mot_juice_bottle.items():
    print(f'\n[{key}]')
    if isinstance(val, list):
        for v in val:
            print(f'  • {v}')
    else:
        print(f'  {val}')

print('\n→ Logical anomaly = liquid color does NOT match the fruit tag')

=== juice_bottle Match-of-Thought ===

[interests_of_thought]
  • glass
  • liquid in bottle
  • fruit
  • label
  • black background

[compositional_rules_of_thought]
  • consistency of fruit tag and liquid color

[anomaly_free_caption]
  • cherry juice bottle (red liquid + cherry tag)
  • orange juice bottle (yellow liquid + orange tag)
  • banana juice bottle (milky liquid + banana tag)

[liquid_color_prompts]
  • red liquid
  • yellow liquid
  • milky liquid

[fruit_tag_prompts]
  • cherry
  • orange
  • banana

→ Logical anomaly = liquid color does NOT match the fruit tag


## 3. Memory Bank — Full-data Protocol

For the **full-data** protocol, patch features of all normal training images
are pre-computed with `compute_coreset.py` and stored as `.pt` files.

At inference time, each test patch is matched against this memory bank
(nearest-neighbour search) to compute an anomaly score.

In [4]:
#OWN CODE — load & display the pre-computed full-data memory-bank tensors
import torch

# Load pre-computed memory bank for juice_bottle (full-data)
clip_bank  = torch.load('memory_bank/mem_patch_feature_clip_juice_bottle.pt',
                        map_location='cpu')
dino_bank  = torch.load('memory_bank/mem_patch_feature_dinov2_juice_bottle.pt',
                        map_location='cpu')
inst_bank  = torch.load('memory_bank/mem_instance_features_multi_stage_juice_bottle.pt',
                        map_location='cpu')

print('Memory bank contents (juice_bottle, full-data):')
print(f'  CLIP patch features   : {clip_bank.shape}  ({clip_bank.nbytes/1e6:.0f} MB)')
print(f'  DINOv2 patch features : {dino_bank.shape}  ({dino_bank.nbytes/1e6:.0f} MB)')
print(f'  Instance features     : {inst_bank.shape}  ({inst_bank.nbytes/1e6:.0f} MB)')
print()
print('For the 4-shot protocol, the memory bank is built at runtime')
print('from only 4 normal images — no pre-computation needed.')

Memory bank contents (juice_bottle, full-data):
  CLIP patch features   : torch.Size([339968, 4096])  (5570 MB)
  DINOv2 patch features : torch.Size([339968, 4096])  (5570 MB)
  Instance features     : torch.Size([350, 4096])  (6 MB)

For the 4-shot protocol, the memory bank is built at runtime
from only 4 normal images — no pre-computation needed.


## 4. Run Evaluation — 4-shot Only

This cell runs the MVTec LOCO **4-shot** evaluation only. This is the reduced setting used for the submitted notebook because the full 1-shot/2-shot/4-shot/full-data sweep is GPU-heavy and takes too long to rerun interactively.

The evaluation still uses the paper dataset/model setup for the 4-shot protocol across all five LOCO categories. To avoid overwriting the repository-level `outputs/` folder, this notebook writes artifacts to `demo/Group6-full-01-results/MVTec_LOCO_4shot/`.


In [5]:
#OWN CODE — run MVTec LOCO 4-shot evaluation and stream output live
from pathlib import Path
import os
import re
import subprocess
import sys

evaluation_output_dir = Path("demo/Group6-full-01-results/MVTec_LOCO_4shot")
log_dir = evaluation_output_dir / "logs" / "4-shot"
result_path = evaluation_output_dir / "results.txt"
log_dir.mkdir(parents=True, exist_ok=True)

dataset_path = os.environ.get("DATASET_PATH", str(Path.cwd() / "datasets" / "MVTec_LOCO"))
python_bin = os.environ.get("PYTHON_BIN", sys.executable)
categories = [
    ("breakfast_box", "Breakfast Box"),
    ("juice_bottle", "Juice Bottle"),
    ("pushpins", "Pushpins"),
    ("screw_bag", "Screw Bag"),
    ("splicing_connectors", "Splicing Connectors"),
]

print("MVTec LOCO 4-shot evaluation")
print(f"Dataset path: {dataset_path}")
print(f"Notebook-only output dir: {evaluation_output_dir}")
print()

metric_values = []
for category, display_name in categories:
    log_path = log_dir / f"{category}.log"
    command = [
        python_bin, "-u", "evaluation.py",
        "--module_path", "model_ensemble_few_shot",
        "--category", category,
        "--dataset_path", dataset_path,
        "--k_shot", "4",
    ]
    print(f"[4-shot] {display_name}")
    print(" ".join(command))
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        live_tail = []
        for line in process.stdout:
            log_file.write(line)
            log_file.flush()
            live_tail.append(line.rstrip())
            live_tail = live_tail[-5:]
        return_code = process.wait()
    if return_code != 0:
        print("Last log lines before failure:")
        print("\n".join(live_tail))
        raise RuntimeError(f"Evaluation failed for {category} with return code {return_code}")

    log_text = log_path.read_text(encoding="utf-8", errors="replace")
    final_rows = [line for line in log_text.splitlines() if re.search(rf"\|\s*{re.escape(category)}\s*\|", line)]
    if not final_rows:
        raise RuntimeError(f"Could not find metric row for {category} in {log_path}")
    cells = [cell.strip() for cell in final_rows[-1].strip().strip("|").split("|")]
    f1, auroc = float(cells[2]), float(cells[3])
    metric_values.append((display_name, f1, auroc))
    print(f"  F1-max={f1:.1f}, AUROC={auroc:.1f}")
    print(f"  log: {log_path}")

avg_f1 = sum(v[1] for v in metric_values) / len(metric_values)
avg_auroc = sum(v[2] for v in metric_values) / len(metric_values)

headers1 = ["Protocol"]
for name, _, _ in metric_values:
    headers1.extend([name, ""])
headers1.extend(["Average", ""])
headers2 = [""] + [metric for _ in metric_values + [("Average", avg_f1, avg_auroc)] for metric in ["F1-max", "AUROC"]]
row = ["4-shot"]
for _, f1, auroc in metric_values:
    row.extend([f"{f1:.1f}", f"{auroc:.1f}"])
row.extend([f"{avg_f1:.1f}", f"{avg_auroc:.1f}"])
table_rows = [headers1, headers2, row]
widths = [max(len(str(r[i])) for r in table_rows) for i in range(len(headers1))]

def line(items):
    return "| " + " | ".join(str(item).ljust(widths[i]) for i, item in enumerate(items)) + " |"

def sep():
    return "|-" + "-|-".join("-" * width for width in widths) + "-|"

evaluation_result_text = "\n".join([
    "Table 9 subset. Image-level F1-max and AUROC results on MVTec LOCO, 4-shot protocol only.",
    "",
    line(headers1),
    sep(),
    line(headers2),
    sep(),
    line(row),
]) + "\n"
result_path.write_text(evaluation_result_text, encoding="utf-8")

print()
print(f"Saved 4-shot result file: {result_path}")
print(evaluation_result_text)


MVTec LOCO 4-shot evaluation
Dataset path: /home/gaya6/LogSAD/datasets/MVTec_LOCO
Notebook-only output dir: demo/Group6-full-01-results/MVTec_LOCO_4shot

[4-shot] Breakfast Box
/home/gaya6/miniconda3/envs/logsad/bin/python -u evaluation.py --module_path model_ensemble_few_shot --category breakfast_box --dataset_path /home/gaya6/LogSAD/datasets/MVTec_LOCO --k_shot 4
  F1-max=89.9, AUROC=94.4
  log: demo/Group6-full-01-results/MVTec_LOCO_4shot/logs/4-shot/breakfast_box.log
[4-shot] Juice Bottle
/home/gaya6/miniconda3/envs/logsad/bin/python -u evaluation.py --module_path model_ensemble_few_shot --category juice_bottle --dataset_path /home/gaya6/LogSAD/datasets/MVTec_LOCO --k_shot 4
  F1-max=88.2, AUROC=84.3
  log: demo/Group6-full-01-results/MVTec_LOCO_4shot/logs/4-shot/juice_bottle.log
[4-shot] Pushpins
/home/gaya6/miniconda3/envs/logsad/bin/python -u evaluation.py --module_path model_ensemble_few_shot --category pushpins --dataset_path /home/gaya6/LogSAD/datasets/MVTec_LOCO --k_shot 4
 

## 5. Evaluation Results vs. Paper

The comparison below uses `evaluation_result_text`, which is created by the 4-shot evaluation cell above. If this notebook is opened without rerunning Section 4, the cell falls back to the notebook-specific 4-shot result file in `demo/Group6-full-01-results/MVTec_LOCO_4shot/results.txt`.


In [6]:
#OWN CODE — display the 4-shot result produced by the evaluation cell above
from pathlib import Path

results_path = Path("demo/Group6-full-01-results/MVTec_LOCO_4shot/results.txt")
if "evaluation_result_text" not in globals():
    evaluation_result_text = results_path.read_text(encoding="utf-8")

print(evaluation_result_text)


Table 9 subset. Image-level F1-max and AUROC results on MVTec LOCO, 4-shot protocol only.

| Protocol | Breakfast Box |       | Juice Bottle |       | Pushpins |       | Screw Bag |       | Splicing Connectors |       | Average |       |
|----------|---------------|-------|--------------|-------|----------|-------|-----------|-------|---------------------|-------|---------|-------|
|          | F1-max        | AUROC | F1-max       | AUROC | F1-max   | AUROC | F1-max    | AUROC | F1-max              | AUROC | F1-max  | AUROC |
|----------|---------------|-------|--------------|-------|----------|-------|-----------|-------|---------------------|-------|---------|-------|
| 4-shot   | 89.9          | 94.4  | 88.2         | 84.3  | 81.4     | 82.5  | 83.0      | 81.3  | 84.7                | 88.6  | 85.4    | 86.2  |



In [7]:
#OWN CODE — compare paper 4-shot AUROC with the result generated by the evaluation cell above
import pandas as pd
from pathlib import Path

if "evaluation_result_text" not in globals():
    evaluation_result_text = Path("demo/Group6-full-01-results/MVTec_LOCO_4shot/results.txt").read_text(encoding="utf-8")

paper = pd.DataFrame({
    "Protocol": ["4-shot"],
    "Breakfast Box": [94.4],
    "Juice Bottle": [84.3],
    "Pushpins": [82.5],
    "Screw Bag": [81.5],
    "Splicing Con.": [88.6],
    "Average AUROC": [86.3],
}).set_index("Protocol")

categories = ["Breakfast Box", "Juice Bottle", "Pushpins", "Screw Bag", "Splicing Con.", "Average AUROC"]
rows = []
for line in evaluation_result_text.splitlines():
    if not line.startswith("| 4-shot"):
        continue
    cells = [cell.strip() for cell in line.strip().strip("|").split("|")]
    auroc_values = [float(cells[i]) for i in [2, 4, 6, 8, 10, 12]]
    rows.append({"Protocol": cells[0], **dict(zip(categories, auroc_values))})

reproduced = pd.DataFrame(rows).set_index("Protocol")

print("=== Paper Table 9 — 4-shot AUROC ===")
print(paper.to_string())
print()
print("=== Ours — 4-shot AUROC parsed from notebook evaluation result ===")
print(reproduced.to_string())
print()
print("Δ (Ours - Paper):")
print((reproduced - paper).to_string())


=== Paper Table 9 — 4-shot AUROC ===
          Breakfast Box  Juice Bottle  Pushpins  Screw Bag  Splicing Con.  Average AUROC
Protocol                                                                                
4-shot             94.4          84.3      82.5       81.5           88.6           86.3

=== Ours — 4-shot AUROC parsed from notebook evaluation result ===
          Breakfast Box  Juice Bottle  Pushpins  Screw Bag  Splicing Con.  Average AUROC
Protocol                                                                                
4-shot             94.4          84.3      82.5       81.3           88.6           86.2

Δ (Ours - Paper):
          Breakfast Box  Juice Bottle  Pushpins  Screw Bag  Splicing Con.  Average AUROC
Protocol                                                                                
4-shot              0.0           0.0       0.0       -0.2            0.0           -0.1
